In [1]:
import polars as pl
# lectura de la muestra de 20k registros
df = pl.read_parquet("../02_data/raw/muestra_20k_2026-11-25.parquet")
print(df.shape)

(20000, 47)


In [2]:
# conteo de campos nulos por columna
print(df.null_count())


shape: (1, 47)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ nombre_en ┆ nit_entid ┆ codigo_en ┆ departame ┆ … ┆ valor_fac ┆ valor_pag ┆ valor_pen ┆ valor_pe │
│ tidad     ┆ ad        ┆ tidad     ┆ nto       ┆   ┆ turado    ┆ ado       ┆ diente_de ┆ ndiente_ │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ _pago     ┆ de_ejecu │
│ u32       ┆ u32       ┆ u32       ┆ u32       ┆   ┆ u32       ┆ u32       ┆ ---       ┆ cion     │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆ u32       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆ u32      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 0         ┆ 0         ┆ 0         ┆ 0         ┆ … ┆ 0         ┆ 0         ┆ 0         ┆ 0        │
└───────────┴───────────┴───────────┴───────────┴───┴───────────┴───────────

In [3]:
# textos de relleno
print(df["ciudad"].value_counts(sort=True))
print(df["sector"].value_counts(sort=True))
print(df["habilita_pago_adelantado"].value_counts(sort=True))
print(df["destino_gasto"].value_counts(sort=True))
print(df["estado_contrato"].value_counts(sort=True))



shape: (511, 2)
┌──────────────┬───────┐
│ ciudad       ┆ count │
│ ---          ┆ ---   │
│ str          ┆ u32   │
╞══════════════╪═══════╡
│ No Definido  ┆ 4261  │
│ Bogotá       ┆ 3950  │
│ Cali         ┆ 1423  │
│ Medellín     ┆ 899   │
│ Barranquilla ┆ 451   │
│ …            ┆ …     │
│ Frontino     ┆ 1     │
│ San Bernardo ┆ 1     │
│ Toro         ┆ 1     │
│ Sativanorte  ┆ 1     │
│ La Salina    ┆ 1     │
└──────────────┴───────┘
shape: (24, 2)
┌─────────────────────────────────┬───────┐
│ sector                          ┆ count │
│ ---                             ┆ ---   │
│ str                             ┆ u32   │
╞═════════════════════════════════╪═══════╡
│ Servicio Público                ┆ 5045  │
│ No aplica/No pertenece          ┆ 3060  │
│ Salud y Protección Social       ┆ 2882  │
│ Educación Nacional              ┆ 1109  │
│ Trabajo                         ┆ 960   │
│ …                               ┆ …     │
│ Tecnologías de la Información … ┆ 149   │
│ interior      

Hay resultados faltantes, a pesar de haber sido llenados como texto (ejemplo: No Definido, No aplica) y también hay un hallazgo relevante: hay variables que tienen errores de escritura (typo).

Para el caso de la variable ciudad, aparecen más datos faltantes (No Definido: 4.261) que incluso la ciudad Bogotá. De 20.000 datos, el 21,3% son datos faltantes

Para el caso de sector, a pesar de que Servicio Público aparece 5045, no es claro a qué corresponde 'Servicio Público', porque no coincide con los sectores administrativos oficiales; queda pendiente verificarlo. Los demás sectores que aparecen sí existen. Además No aplica/No pertenece aparece 3060, es decir que un 15,3% son datos faltantes

Para la variable habilita_pago_adelantado el 8,8% son datos faltantes

Para la variable destino_gasto hay 0,7% de datos faltantes.

Para la variable de estado_contrato todos los datos son estados válidos y distintos a pesar de que hay unos escritos en mayúscula y otros no. En la limpieza habrá que unificarlos. Pero hay una diferenciación técnica y legal entre ellos:
Estados principales que son los estados definitivos es decir si está activo, frenado o finalizado:
- En ejecución
- Terminado
- Cerrado
Estados precontractuales:
- Aprobado
- Enviado proveedor
- En aprobación
Estados de modificación (contrato en curso):
- Suspendido (se congela la ejecución)
- Modificado (el contrato original sufrió un cambio en sus condiciones iniciales)
- Cedido (el contratista le transfirió sus derechos y obligaciones a un tercero)

Por eso se debe ser muy cuidadoso, pues un contrato puede estar "modificado" o "cedido" pero su estado operativo sigue siendo "en ejecución".

Hallazgo al investigar: id_contrato es el identificador único universal de un contrato firmado en la plataforma. En cambio id identifica una transacción o reporte específico. 

Usé ":id", que es diferente de los anteriores, este lo crea la plataforma de datos abiertos para identificar cada fila de la tabla, no identifica transacción ni un reporte, solo es una posición de la fila y, por eso, sirve para muestrear, pues no tiene relación con el contenido del contrato.

Anticipos: ¿se puede calcular la bandera "anticipos en riesgo"?

En la muestra, habilita_pago_adelantado tiene:
- No: 18.230 contratos (91,2 %)
- No Definido: 1.760 contratos (8,8 %)
- Si: 10 contratos (0,05 %)

si la proporción se mantiene, de los 5,73 millones de contratos unos 2.900 tendrían anticipo (0,05 % × 5.730.000 ≈ 2.865). Con solo 10 casos en la muestra la estimación es poco precisa: el número real podría estar aproximadamente entre 1.400 y 5.300 contratos.

- Con la muestra no se puede analizar esta bandera: 10 casos son muy pocos para encontrar patrones.
- En la base completa sí habría suficientes casos (miles), así que la bandera se mantiene, pero se calculará con la base completa.
- Es un grupo pequeño de contratos, pero puede involucrar mucho dinero, porque los anticipos suelen darse en contratos grandes. Queda pendiente verificar el valor de esos contratos.
- El 8,8 % de "No Definido" es una limitación: en esos contratos no se sabe si hubo anticipo o no, así que la bandera podría subestimar los casos reales.
- Esta es una decisión basada en los datos, pues antes de explorarlos se suponía que los anticipos eran comunes, y la muestra muestra que son raros.


In [4]:
# valores 0
print((df["valor_del_contrato"] == "0").sum())
print((df["valor_pagado"] == "0").sum())
print((df["valor_facturado"] == "0").sum())
print((df["valor_de_pago_adelantado"] == "0").sum())

157
9864
8524
19987


1. valor_de_pago_adelantado
19.987 ceros es consistente con la idea que ya habíamos encontrado, pues casi nadie recibe anticipo. A pesar de eso no coincide con los 10 donde sí estaba habilitado el pago adelantado. Hay una inconsistencia. Valdría la pena revisar el estado del contrato en estos casos.

2. valor_del_contrato
157 en cero, pero sí existen casos legítimos de contratos por valor 0:
- convenios interadministrativos o de asociación sin aportes dinerarios
- contratos de comodato
- contratos de cuantía indeterminada o remunerados por terceros
- contratos de donación o cooperación

3. valor_pagado (9.864) y valor facturado (8.524)
¿dependen del estado del contrato?

In [5]:
# comprobación hipótesis 
# valor_del contrato según modalidad de contratación
tabla_filtrada = df.filter(pl.col("valor_del_contrato") =="0")
print(tabla_filtrada["justificacion_modalidad_de"].value_counts(sort=True))

shape: (12, 2)
┌─────────────────────────────────┬───────┐
│ justificacion_modalidad_de      ┆ count │
│ ---                             ┆ ---   │
│ str                             ┆ u32   │
╞═════════════════════════════════╪═══════╡
│ Contratos o convenios Interadm… ┆ 63    │
│ Prestamo de uso                 ┆ 36    │
│ Arrendamiento de inmuebles      ┆ 18    │
│ Regla aplicable                 ┆ 13    │
│ Servicios profesionales y apoy… ┆ 7     │
│ …                               ┆ …     │
│ Presupuesto inferior al 10% de… ┆ 3     │
│ Ley 1150 de 2007                ┆ 3     │
│ Operaciones de Crédito Público  ┆ 2     │
│ Artículo 49, Decreto 1510 de 2… ┆ 2     │
│ Contratos o convenios Interadm… ┆ 1     │
└─────────────────────────────────┴───────┘


Los casos que llaman la atención son:
- arrendamiento de inmuebles (18): el arrendamiento por ley tiene precio.
- Servicios profesionales y de apoyo a la gestión (7): en principio son remunerados, pero podría tratarse de una pasantía o judicaturas ad honorem.
- Presupuesto inferior al 10% de la menor cuantía (3): la mínima cuantía aplica para compra de bienes o servicios con precio.
- Artículo 49, Decreto 1510 de 2013 (2):
este artículo regula la contratación de servicios de salud mediante un régimen especial/ contratación directa.


In [7]:
# comprobación hipótesis revisión del estado del contrato según valor pagado == 0
filtrada_estado = df.filter(pl.col("valor_pagado") == "0")
print(filtrada_estado["estado_contrato"].value_counts(sort=True))

# comprobación hipótesis revisión del estado del contrato según valor facturado == 0
filtrada_facturado = df.filter(pl.col("valor_facturado") == "0")
print(filtrada_facturado["estado_contrato"].value_counts(sort=True))

filtrada_ambos = df.filter(
    (pl.col("valor_facturado") == "0") &
    (pl.col("valor_pagado") == "0")
)
print(filtrada_ambos["estado_contrato"].value_counts(sort=True))

shape: (9, 2)
┌───────────────────┬───────┐
│ estado_contrato   ┆ count │
│ ---               ┆ ---   │
│ str               ┆ u32   │
╞═══════════════════╪═══════╡
│ En ejecución      ┆ 3605  │
│ Modificado        ┆ 2159  │
│ Cerrado           ┆ 2052  │
│ terminado         ┆ 1201  │
│ Aprobado          ┆ 588   │
│ enviado Proveedor ┆ 155   │
│ En aprobación     ┆ 64    │
│ cedido            ┆ 25    │
│ Suspendido        ┆ 15    │
└───────────────────┴───────┘
shape: (9, 2)
┌───────────────────┬───────┐
│ estado_contrato   ┆ count │
│ ---               ┆ ---   │
│ str               ┆ u32   │
╞═══════════════════╪═══════╡
│ En ejecución      ┆ 2847  │
│ Cerrado           ┆ 2052  │
│ Modificado        ┆ 1791  │
│ terminado         ┆ 1014  │
│ Aprobado          ┆ 570   │
│ enviado Proveedor ┆ 155   │
│ En aprobación     ┆ 64    │
│ cedido            ┆ 16    │
│ Suspendido        ┆ 15    │
└───────────────────┴───────┘
shape: (9, 2)
┌───────────────────┬───────┐
│ estado_contrato   ┆ count 

Se encuentran 3.253 contratos con estado "Cerrado" (2.052) y "terminado" (1.201) que tienen 0 en valor_pagado. De ellos, 3.066 tienen en 0 tanto valor_pagado como valor_facturado (2.052 cerrados y 1.014 terminados), y los 187 restantes tienen facturas registradas pero ningún pago.

Además, la tabla de contratos con ambas columnas en 0 es idéntica a la de valor_facturado = 0: todo contrato sin facturas registradas tampoco tiene pagos registrados, lo cual es lógico porque en SECOP II no se registra un pago sin una factura previa.

Esto es raro pues contratos en dicho estado deberían tener pagos registrados. Una posible explicación es que no necesariamente falta el pago: en SECOP II el pago solo queda registrado cuando la entidad marca la factura como pagada en el plan de pagos. Distintas auditorías han documentado que muchas entidades no lo hacen. Por eso el cero puede ser un subregistro. Los 187 contratos con factura pero sin pago son los casos más claros de este posible subregistro.

Esto implica que valor_pagado = 0 no se puede utilizar para los indicadores de riesgo. Pero sí se puede calcular el % de contratos terminados con pagos en cero por entidad. Se mide la transparencia y organización frente a este sistema dentro de la entidad.

## Limitaciones de calidad de los datos de SECOP II y cómo afectan el análisis de riesgo

### 1. Contexto

Usar SECOP II no es opcional: las entidades estatales están obligadas a publicar allí los documentos y actos de sus procesos de contratación dentro de los tres días siguientes a su expedición (Decreto 1082 de 2015, art. 2.2.1.1.1.7.1), y la información pública debe ser completa, oportuna y veraz (Ley 1712 de 2014, arts. 3, 9 y 10). Sin embargo, la norma por sí sola no garantiza que los campos del dataset estén completos, actualizados o validados.

SECOP II es una **fuente administrativa**, no un registro contable que se actualiza solo: muchos campos cambian únicamente cuando la **entidad** realiza una acción en la plataforma (publicar el acta de terminación, marcar una factura como pagada, cerrar el expediente). El **proveedor** solo interviene en algunos pasos, como cargar las facturas. Por eso existe el riesgo de **confundir ausencia de registro con ausencia del hecho**.

### 2. Problemas documentados por variable

| Variable | Problema | Por qué ocurre |
|---|---|---|
| `valor_pagado` | Puede quedar en 0 aunque el pago sí se hizo | Solo sube cuando la entidad marca la factura como pagada en el plan de pagos |
| `valor_facturado` | Puede quedar en 0 | Depende de que el proveedor cargue la factura y la entidad la gestione |
| `valor_pendiente_de_pago` | Puede quedar inconsistente | Depende de que se actualicen facturas, pagos y modificaciones |
| `valor_amortizado` | Puede quedar en 0 o desactualizado | Depende de que se registre el plan de amortización del anticipo |
| `estado_contrato` | Contratos terminados que siguen "En ejecución" | El estado no es un campo independiente: resulta de los documentos que la entidad publica; si no publica la terminación o el cierre, el estado queda desfasado |
| Fechas y duración | Fechas incoherentes, prórrogas no reflejadas | Diligenciamiento incorrecto y modificaciones no registradas a tiempo |
| `ciudad`, `sector` | Valores "No Definido" y "No aplica/No pertenece" | Campos que no se diligencian o no aplican |
| Proveedores | Errores de digitación en nombres | Conviene identificar al proveedor por tipo y número de documento, no por el nombre |
| BPIN, origen y destino del gasto | Incompletos o inconsistentes | Dependen de otros sistemas del Estado |

**Evidencia verificada en auditorías del IDEAM:**
- La auditoría interna de gestión contractual de 2026 (informe del 6 de octubre de 2026, contratos de 2023 a 2025) encontró **258 contratos, por $15.332.567.348**, que ya habían terminado y que la propia entidad reporta como ejecutados, pero que **seguían en estado "En ejecución" en SECOP II**, algunos hasta 35 meses después de terminar. El informe concluye que la información publicada no solo es incompleta sino incorrecta (págs. 19-20). Para el cruce, la auditoría usó el mismo dataset de este proyecto, "SECOP II – Contratos Electrónicos" (pág. 5).
- La auditoría interna del proceso de gestión jurídica y contractual de 2021 (informe del 9 de diciembre de 2021) encontró **17 contratos de 2021 sin información de ejecución y supervisión en SECOP II** (informes, cuentas de cobro o facturas), y señaló que no existía un procedimiento para aprobar a tiempo esos registros en la plataforma (Hallazgo 3, pág. 34).

### 3. Lo que encontré en mi muestra (20.000 contratos)

- **Pagos:** 3.253 contratos "Cerrado" o "terminado" tienen `valor_pagado` = 0. De ellos, 3.066 no tienen ni facturas ni pagos registrados, y 187 tienen facturas pero ningún pago. Esto coincide con lo documentado: el cero refleja falta de registro, no necesariamente falta de pago.
- **Anticipos:** solo 10 contratos (0,05 %) tienen anticipo habilitado, y el 8,8 % de `habilita_pago_adelantado` es "No Definido".
- **Ubicación y sector:** el 21,3 % de `ciudad` es "No Definido" y el 15,3 % de `sector` es "No aplica/No pertenece".
- **Estado:** `estado_contrato` mezcla mayúsculas y minúsculas ("terminado", "cedido"), lo que habrá que unificar en la limpieza.

### 4. Implicaciones para el análisis

- Distinguir siempre entre **dato faltante**, **cero registrado** y **categoría "no aplica"**.
- `valor_pagado` = 0 **no se usa como bandera de riesgo directa**; sirve como alerta, no como conclusión.
- `estado_contrato` debe usarse con cuidado: un contrato vencido que sigue "En ejecución" es una señal de **cierre administrativo incompleto**, no prueba de que el contrato siga ejecutándose.
- Indicadores posibles por entidad: % de contratos terminados sin pagos registrados, % de contratos vencidos que siguen "En ejecución" y tiempo transcurrido desde la fecha de fin. Estos miden **transparencia y calidad del registro**, que también es una obligación legal.
- Validaciones en la limpieza: fechas coherentes, duración positiva y valor mayor o igual a cero.
- Reportar siempre la fecha de corte de los datos y la proporción de registros incompletos.
- Trabajo futuro: existen otros datasets de SECOP II (por ejemplo, facturas y plan de pagos) que podrían cruzarse con los contratos para confirmar si hubo pagos.

### 5. Fuentes

- IDEAM, Oficina de Control Interno. *Informe ejecutivo de auditoría: gestión contractual* (6 de octubre de 2026). https://www.ideam.gov.co/sites/default/files/transparencia/controlinterno/informe_final_de_la_auditoria_de_gestion_contractual_2026.pdf
- IDEAM, Oficina de Control Interno. *Informe de auditoría interna, proceso de gestión jurídica y contractual* (9 de diciembre de 2021). https://www.ideam.gov.co/sites/default/files/transparencia/controlinterno/informe_de_auditoria_interna_proceso_de_gestion_juridica_y_contractual.pdf
- Decreto 1082 de 2015, art. 2.2.1.1.1.7.1 (publicidad en el SECOP).
- Ley 1712 de 2014 (transparencia y acceso a la información pública), arts. 3, 9 y 10.
- Colombia Compra Eficiente, Circular Externa 002 de 2023 (cierre de expedientes electrónicos en SECOP II), citada en la auditoría del IDEAM de 2026.
- *Por verificar:* la tabla de la sección 2 resume una búsqueda hecha con Perplexity; las filas que no aparecen en las auditorías anteriores todavía necesitan su fuente directa (guías de SECOP II de Colombia Compra Eficiente y otras auditorías).

### 6. Reflexión personal

Este análisis se hace con datos de Colombia, un país donde la contratación pública ha estado en el centro de grandes escándalos, como el carrusel de la contratación en Bogotá, Reficar, Odebrecht, Centros Poblados (MinTIC) o la UNGRD. Eso no significa que las entidades con datos incompletos sean corruptas pues los datos no evidencian intenciones ocultas. Pero sí explica por qué vale la pena vigilar la contratación con datos abiertos.

Registrar la información en SECOP II es una obligación legal, no una opción, así que las inconsistencias deben tomarse en serio pues reflejan el nivel de transparencia de cada entidad. Queda abierta la pregunta de por qué ocurren si es descuido, falta de capacitación, rotación del personal o sobrecarga de trabajo, y sería interesante investigar qué capacidades tienen las entidades que más incumplen.